# Hinglish Sarcasm Detector — Training

Fine-tune a multilingual transformer (**MuRIL** — built by Google for Indian languages incl. Hinglish) for **binary sarcasm detection**.

## 1. Install (run once)

In [1]:
!pip install -q transformers datasets torch scikit-learn pandas numpy

## 2. Imports

In [2]:
import re
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer, DataCollatorWithPadding)

print("GPU:", torch.cuda.is_available())

GPU: True


## 3. Load dataset

Point `CSV_PATH` at your Hinglish sarcasm CSV. It must have a **text** column and a **binary label** column (0 = not sarcasm, 1 = sarcasm).

The cell auto-detects common column names. If yours differ, set `TEXT_COL` / `LABEL_COL` manually.

In [3]:
# --- 1. Sarcasm dataset (label 1 = sarcasm, 0 = not) ---
sarc = pd.read_csv("sarcasm_hinghlish_dataset.csv")[["text", "label"]].dropna()
sarc["label"] = sarc["label"].astype(int)
print("Sarcasm file:", sarc.shape, "| counts:", dict(sarc["label"].value_counts()))

# --- 2. Emotion dataset -> genuine conversational NON-sarcasm negatives (label 0) ---
# WHY: the sarcasm file's "not sarcasm" class is mostly news/political tweets, so the
# model learned "conversational => sarcasm" and false-flags normal chat. We add genuine
# emotional Hinglish (neutral/joy/love/admiration) as extra label-0 examples to fix this.
try:
    emo = pd.read_csv("mlt_hinghlish_dataset.csv").rename(columns={"hinglish_genz_text": "text"})
    emo = emo[["text", "label"]].dropna()
    SAFE = ["neutral", "joy", "love", "admiration"]   # least likely to be sarcastic
    emo = emo[emo["label"].str.lower().isin(SAFE)]
    emo = emo.sample(n=min(4000, len(emo)), random_state=42).copy()
    emo["label"] = 0                                  # all -> not sarcasm
    print(f"Added {len(emo)} genuine conversational negatives from emotion data")
    df = pd.concat([sarc, emo], ignore_index=True)
except FileNotFoundError:
    print("mlt_hinghlish_dataset.csv not found -- using sarcasm data only")
    df = sarc

# clean + shuffle
df["text"] = df["text"].astype(str)
df = df.drop_duplicates(subset="text").dropna()
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

print("
Total after mixing:", df.shape)
print(df["label"].value_counts())
df.head()

Columns: ['text', 'label']
Using text column: text | label column: label
(9593, 2)
label
1    5544
0    4049
Name: count, dtype: int64


,text,label
0,Ckt News: S Ajmal Sohail khan k BaaD Ab Umar A...,0
1,Online resources se bahut kuch seekh rahe,0
2,Aab maaf kar dena chahiye-or azad bhi kar do b...,0
3,Bahan k sath-sath biwi ko v nhi mante.Parinam ...,0
4,sakta ham talk ke baare mein something else? k...,0


## 4. Hinglish normalization

Light cleaning only — **keep emojis and `/s`**, they are strong sarcasm signals. Just tame char-elongation (`brooo → broo`) and repeated punctuation. The transformer's own tokenizer handles the rest.

In [4]:
def normalize_hinglish(text):
    text = str(text)
    text = re.sub(r'(.)\1{2,}', r'\1\1', text)     # brooo -> broo
    text = re.sub(r'([!?.]){2,}', r'\1\1', text)   # !!!! -> !!
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['text'] = df['text'].apply(normalize_hinglish)
df.head()

,text,label
0,Ckt News: S Ajmal Sohail khan k BaaD Ab Umar A...,0
1,Online resources se bahut kuch seekh rahe,0
2,Aab maaf kar dena chahiye-or azad bhi kar do b...,0
3,Bahan k sath-sath biwi ko v nhi mante.Parinam ...,0
4,sakta ham talk ke baare mein something else? k...,0


## 5. Train / val split (stratified, like the repo's 80/10/10 → here 85/15)

In [5]:
train_df, val_df = train_test_split(
    df, test_size=0.15, random_state=42, stratify=df['label'])
print("train:", len(train_df), " val:", len(val_df))

train: 8154  val: 1439


## 6. Tokenizer + model

**MuRIL** (`google/muril-base-cased`) — pretrained on 17 Indian languages *and* their transliterated (Roman/Hinglish) forms. Best fit for code-mixed text.

> Alternatives: `bert-base-multilingual-cased` (mBERT) or `ai4bharat/indic-bert`. Just change `MODEL_NAME`.

In [6]:
MODEL_NAME = "google/muril-base-cased"
MAX_LEN = 128

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

config.json:   0%|          | 0.00/411 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/206 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/3.16M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/113 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/953M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: google/muril-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params w

## 7. Datasets

In [7]:
class SarcasmDataset(torch.utils.data.Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.enc = tokenizer(list(texts), truncation=True, max_length=max_len)
        self.labels = list(labels)
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, i):
        item = {k: v[i] for k, v in self.enc.items()}
        item['labels'] = self.labels[i]
        return item

train_ds = SarcasmDataset(train_df['text'], train_df['label'], tokenizer, MAX_LEN)
val_ds   = SarcasmDataset(val_df['text'],   val_df['label'],   tokenizer, MAX_LEN)
collator = DataCollatorWithPadding(tokenizer)

## 8. Metrics

In [8]:
def compute_metrics(pred):
    preds = pred.predictions.argmax(-1)
    return {
        'accuracy': accuracy_score(pred.label_ids, preds),
        'f1': f1_score(pred.label_ids, preds, average='macro'),
    }

## 9. Fine-tune

Repo config: lr `5e-5`, batch `32`, weight_decay `0.01`, warmup `500`, label_smoothing `0.1`, AdamW.
Epochs reduced 20 → **4** with early best-model selection (20 would badly overfit ~9k samples — the exact mistake we hit before).

In [14]:
args = TrainingArguments(
    output_dir="hinglish_out",
    num_train_epochs=4,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    learning_rate=5e-5,
    weight_decay=0.01,
    warmup_steps=500,
    label_smoothing_factor=0.1,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_steps=50,
    fp16=torch.cuda.is_available(),
)

trainer = Trainer(
    model=model, args=args,
    train_dataset=train_ds, eval_dataset=val_ds,
    processing_class=tokenizer, data_collator=collator,
    compute_metrics=compute_metrics,
)
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.220364,0.309879,0.945101,0.943737
2,0.230752,0.331123,0.943016,0.941859
3,0.223071,0.302717,0.949965,0.948209
4,0.216493,0.307515,0.949270,0.947819


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1020, training_loss=0.22745813809189142, metrics={'train_runtime': 1234.583, 'train_samples_per_second': 26.419, 'train_steps_per_second': 0.826, 'total_flos': 932728691251200.0, 'train_loss': 0.22745813809189142, 'epoch': 4.0})

## 10. Evaluate

In [15]:
print(trainer.evaluate())

preds = trainer.predict(val_ds).predictions.argmax(-1)
print(classification_report(val_df['label'], preds,
      target_names=['not sarcasm', 'sarcasm']))

Training Loss,Validation Loss,Epoch,Accuracy,F1
0.216493,0.302717,4,0.949965,0.948209


{'eval_loss': 0.30271655321121216, 'eval_accuracy': 0.9499652536483669, 'eval_f1': 0.9482088442066368}


              precision    recall  f1-score   support

 not sarcasm       0.97      0.91      0.94       607
     sarcasm       0.94      0.98      0.96       832

    accuracy                           0.95      1439
   macro avg       0.95      0.94      0.95      1439
weighted avg       0.95      0.95      0.95      1439



## 10b. Calibrate decision threshold

Default threshold 0.5 was cutting through borderline sarcasm (many true sarcasm sat at 0.43-0.49). Instead of guessing, we find the threshold that **maximises macro-F1 on the validation set**. All test cells below then use this `THRESHOLD`.

In [ ]:
import numpy as np
from sklearn.metrics import classification_report

# validation probabilities for the positive (sarcasm) class
val_logits = trainer.predict(val_ds).predictions
val_probs  = torch.softmax(torch.tensor(val_logits), -1)[:, 1].numpy()
y_true     = val_df['label'].values

best_t, best_f1 = 0.5, 0.0
for t in np.arange(0.20, 0.61, 0.01):
    f1 = f1_score(y_true, (val_probs >= t).astype(int), average='macro')
    if f1 > best_f1:
        best_f1, best_t = f1, t

THRESHOLD = round(float(best_t), 2)
print(f"Best threshold = {THRESHOLD}   (macro-F1 = {best_f1:.4f}, default 0.50 F1 was compared)")
print(classification_report(y_true, (val_probs >= THRESHOLD).astype(int),
      target_names=['not sarcasm', 'sarcasm']))

## 11. Test on Hinglish samples (borrowed from the repo — real code-mixed examples)

In [28]:
device = model.device
THRESHOLD = globals().get('THRESHOLD', 0.5)   # set by cell 10b; falls back to 0.5

def predict(s):
    enc = tokenizer(normalize_hinglish(s), return_tensors='pt',
                    truncation=True, max_length=MAX_LEN).to(device)
    with torch.no_grad():
        prob = torch.softmax(model(**enc).logits, -1)[0, 1].item()
    label = 'SARCASM   ' if prob >= THRESHOLD else 'not sarcasm'
    print(f"[{label}] {prob*100:5.1f}%  |  {s}")

for s in [
    "Wah! 2 ghante late ho aur bol rahe ho 'bas 5 min'. Genius!",
    "Bhai kya hi logic lagaya hai tune, Nobel prize milna chahiye! /s",
    "Bohot badhiya service hai aapki, 10 din se call wait pe hai. Amazing.",
    "Kya mast room saaf kiya hai, lag raha hai yahan tsunami aayi thi.",
    "Love this product, sach mein badiya kaam kiya team ne!",
    "Mast weather hai yaar, chalo chai peete hain!",
    "Zabardast yaar, keep it up! Proud of you.",
    "Haan haan tumhare paas toh bohot paisa hai na, hum hi gareeb hain.",
]:
    predict(s)

[SARCASM   ]  95.1%  |  Wah! 2 ghante late ho aur bol rahe ho 'bas 5 min'. Genius!
[SARCASM   ]  95.2%  |  Bhai kya hi logic lagaya hai tune, Nobel prize milna chahiye! /s
[SARCASM   ]  95.2%  |  Bohot badhiya service hai aapki, 10 din se call wait pe hai. Amazing.
[SARCASM   ]  95.2%  |  Kya mast room saaf kiya hai, lag raha hai yahan tsunami aayi thi.
[not sarcasm]  26.1%  |  Love this product, sach mein badiya kaam kiya team ne!
[SARCASM   ]  93.9%  |  Mast weather hai yaar, chalo chai peete hain!
[SARCASM   ]  94.8%  |  Zabardast yaar, keep it up! Proud of you.
[SARCASM   ]  95.3%  |  Haan haan tumhare paas toh bohot paisa hai na, hum hi gareeb hain.


## 12. Save model (for the Streamlit app)

Local save below. To deploy easily, also push to Hugging Face Hub (uncomment).

In [ ]:
model.save_pretrained("hinglish-sarcasm-model")
tokenizer.save_pretrained("hinglish-sarcasm-model")
print("Saved to ./hinglish-sarcasm-model")

# --- optional: push to Hugging Face Hub ---
# from huggingface_hub import notebook_login; notebook_login()
# model.push_to_hub("YOUR-USERNAME/hinglish-sarcasm")
# tokenizer.push_to_hub("YOUR-USERNAME/hinglish-sarcasm")

In [30]:
# (text, expected)  1 = sarcasm, 0 = genuine
test_set = [
    ("Wah! 2 ghante late ho aur bol rahe ho 'bas 5 min'. Genius!", 1),
    ("Bhai kya hi logic lagaya, Nobel prize milna chahiye! /s", 1),
    ("10 din se call wait pe hai, bohot badhiya service. Amazing.", 1),
    ("Kya mast room saaf kiya, lagta tsunami aayi thi yahan.", 1),
    ("Haan haan tumhare paas bohot paisa hai, hum hi gareeb hain.", 1),
    ("Wah kya timing hai, exam ke din hi net band ho gaya.", 1),
    ("Tumne toh kamaal kar diya, poora project delete kar diya. Shabaash.", 1),
    ("Bilkul sahi bola tumne, jaise tumhe hi sab pata ho.", 1),
    ("Thank you so much, sach mein bahut help hui aapki.", 0),
    ("Meeting kal 5 baje hai, sabko time pe aana.", 0),
    ("Mujhe ye movie sach mein pasand aayi, story bahut achhi thi.", 0),
    ("Khana tasty tha, chef ko compliment zaroor dena.", 0),
    ("Kal train subah 8 baje hai, ticket book kar li hai.", 0),
    ("Aaj office ka kaam jaldi khatam ho gaya, ghar chalte hain.", 0),
]

correct = 0
for text, exp in test_set:
    enc = tokenizer(normalize_hinglish(text), return_tensors='pt',
                    truncation=True, max_length=MAX_LEN).to(device)
    with torch.no_grad():
        prob = torch.softmax(model(**enc).logits, -1)[0, 1].item()
    TH = globals().get('THRESHOLD', 0.5)
    pred = 1 if prob >= TH else 0
    correct += (pred == exp)
    mark = "OK " if pred == exp else "XX "
    exp_s = "sarcasm" if exp else "genuine"
    print(f"{mark} pred={prob*100:5.1f}%  true={exp_s:8} | {text}")

print(f"Batch accuracy: {correct}/{len(test_set)} = {correct/len(test_set)*100:.1f}%")

OK  pred= 95.1%  true=sarcasm  | Wah! 2 ghante late ho aur bol rahe ho 'bas 5 min'. Genius!
OK  pred= 95.2%  true=sarcasm  | Bhai kya hi logic lagaya, Nobel prize milna chahiye! /s
OK  pred= 95.1%  true=sarcasm  | 10 din se call wait pe hai, bohot badhiya service. Amazing.
OK  pred= 95.2%  true=sarcasm  | Kya mast room saaf kiya, lagta tsunami aayi thi yahan.
OK  pred= 95.3%  true=sarcasm  | Haan haan tumhare paas bohot paisa hai, hum hi gareeb hain.
OK  pred= 95.1%  true=sarcasm  | Wah kya timing hai, exam ke din hi net band ho gaya.
OK  pred= 95.3%  true=sarcasm  | Tumne toh kamaal kar diya, poora project delete kar diya. Shabaash.
OK  pred= 95.3%  true=sarcasm  | Bilkul sahi bola tumne, jaise tumhe hi sab pata ho.
OK  pred= 22.5%  true=genuine  | Thank you so much, sach mein bahut help hui aapki.
OK  pred=  6.2%  true=genuine  | Meeting kal 5 baje hai, sabko time pe aana.
XX  pred= 69.8%  true=genuine  | Mujhe ye movie sach mein pasand aayi, story bahut achhi thi.
XX  pred= 95.1%  t

In [32]:
demo_samples = [
    "Wah! 2 ghante late aur bol rahe ho 'bas 5 min'. Genius!",
    "Bhai kya logic hai, Nobel prize pakka milega! /s",
    "10 din se call wait pe hai... bohot badhiya service aapki.",
    "Haan haan tumhare paas bohot paisa hai, hum hi gareeb hain.",
    "Thank you so much, sach mein bahut help hui aapki.",
    "Mujhe ye movie sach mein pasand aayi, story bahut achhi thi.",
    "Meeting kal 5 baje hai, sabko time pe aana.",
]

print("=" * 62)
print("  HINGLISH SARCASM DETECTOR  (MuRIL, 95% val accuracy)")
print("=" * 62)
for s in demo_samples:
    enc = tokenizer(normalize_hinglish(s), return_tensors='pt',
                    truncation=True, max_length=MAX_LEN).to(device)
    with torch.no_grad():
        prob = torch.softmax(model(**enc).logits, -1)[0, 1].item()
    is_sarc = prob >= globals().get('THRESHOLD', 0.5)
    verdict = "SARCASM" if is_sarc else "GENUINE"
    conf = prob if is_sarc else 1 - prob
    bar = "#" * int(conf * 20)
    print(f"\"{s}\"")
    print(f"     -> {verdict:8} {conf*100:4.1f}%  {bar}")

  HINGLISH SARCASM DETECTOR  (MuRIL, 95% val accuracy)
"Wah! 2 ghante late aur bol rahe ho 'bas 5 min'. Genius!"
     -> SARCASM  86.6%  #################
"Bhai kya logic hai, Nobel prize pakka milega! /s"
     -> SARCASM  95.1%  ###################
"10 din se call wait pe hai... bohot badhiya service aapki."
     -> SARCASM  95.2%  ###################
"Haan haan tumhare paas bohot paisa hai, hum hi gareeb hain."
     -> SARCASM  95.3%  ###################
"Thank you so much, sach mein bahut help hui aapki."
     -> GENUINE  77.5%  ###############
"Mujhe ye movie sach mein pasand aayi, story bahut achhi thi."
     -> SARCASM  69.8%  #############
"Meeting kal 5 baje hai, sabko time pe aana."
     -> GENUINE  93.8%  ##################


In [33]:
fresh_cases = [
    # --- clear sarcasm ---
    ("Kya baat hai, aaj phir se salary late aayi. Wah company wah.", 1),
    ("Tumne toh duniya hila di, 3 din mein ek email bheja. Impressive.", 1),
    ("Perfect, aaj hi laptop kharab hona tha jab submission hai. Superb timing.", 1),
    ("Bahut khoob, tumhari advice se toh main aur fas gaya. Thanks yaar.", 1),
    # --- subtle / contextual sarcasm ---
    ("Nahi nahi, tum bilkul busy ho, main dekh raha hoon 3 ghante se online ho.", 1),
    ("Areh waah, itni jaldi reply? Sirf 2 hafte lag gaye.", 1),
    ("Tumhara plan toh foolproof hai, jaise pichli baar tha.", 1),
    # --- genuine praise (false-positive prone) ---
    ("Bahut mehnat ki hai tumne is project pe, genuinely proud of you.", 0),
    ("Aaj ka khana lajawab tha, maza aa gaya.", 0),
    ("Tumhari presentation clear thi, sabko samajh aayi.", 0),
    # --- genuine complaints (negative but NOT sarcasm) ---
    ("Yaar mera phone bar bar hang ho raha hai, bahut irritating hai.", 0),
    ("Aaj traffic mein 2 ghante nikal gaye, thak gaya hoon.", 0),
    # --- neutral / factual ---
    ("Kal ka match 7 baje start hoga, TV pe aayega.", 0),
    ("Maine grocery le li hai, fridge mein rakh di hai.", 0),
    # --- english-heavy mix ---
    ("Oh wow, another Monday morning meeting. Just what my life needed. 🙃", 1),
    ("Really enjoyed the workshop today, learned a lot, thanks team.", 0),
]

correct = 0
for text, exp in fresh_cases:
    enc = tokenizer(normalize_hinglish(text), return_tensors='pt',
                    truncation=True, max_length=MAX_LEN).to(device)
    with torch.no_grad():
        prob = torch.softmax(model(**enc).logits, -1)[0, 1].item()
    TH = globals().get('THRESHOLD', 0.5)
    pred = 1 if prob >= TH else 0
    correct += (pred == exp)
    mark = "OK " if pred == exp else "XX "
    exp_s = "sarcasm" if exp else "genuine"
    print(f"{mark} pred={prob*100:5.1f}%  true={exp_s:8} | {text}")

print(f"Fresh-test accuracy: {correct}/{len(fresh_cases)} = {correct/len(fresh_cases)*100:.1f}%")

OK  pred= 95.2%  true=sarcasm  | Kya baat hai, aaj phir se salary late aayi. Wah company wah.
OK  pred= 95.3%  true=sarcasm  | Tumne toh duniya hila di, 3 din mein ek email bheja. Impressive.
OK  pred= 95.2%  true=sarcasm  | Perfect, aaj hi laptop kharab hona tha jab submission hai. Superb timing.
OK  pred= 95.3%  true=sarcasm  | Bahut khoob, tumhari advice se toh main aur fas gaya. Thanks yaar.
XX  pred=  9.1%  true=sarcasm  | Nahi nahi, tum bilkul busy ho, main dekh raha hoon 3 ghante se online ho.
OK  pred= 95.2%  true=sarcasm  | Areh waah, itni jaldi reply? Sirf 2 hafte lag gaye.
OK  pred= 95.3%  true=sarcasm  | Tumhara plan toh foolproof hai, jaise pichli baar tha.
XX  pred= 93.1%  true=genuine  | Bahut mehnat ki hai tumne is project pe, genuinely proud of you.
OK  pred=  4.5%  true=genuine  | Aaj ka khana lajawab tha, maza aa gaya.
XX  pred= 95.2%  true=genuine  | Tumhari presentation clear thi, sabko samajh aayi.
XX  pred= 90.4%  true=genuine  | Yaar mera phone bar bar hang ho r